In [ ]:
import itertools
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from LunarLander import *

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True

ENV_NAME = "LunarLander-v2"
SAVE_ROOT = "seed_sweeps_notebook"

In [ ]:
def to_df(summary_rows):
    df = pd.DataFrame(summary_rows).copy()
    if len(df) == 0:
        return df
    numeric_cols = [
        "seed", "size", "depth", "lr", "discount_factor", "episodes",
        "mean_all", "std_all", "min_all", "max_all",
        "mean_last_100", "best_100avg",
        "critic_weight", "entropy_weight_start", "entropy_weight_end",
        "entropy_anneal_episodes", "final_loss", "mean_loss_last_50"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def moving_average(values, window=50):
    values = np.asarray(values, dtype=np.float64)
    if len(values) < window:
        return values
    return np.convolve(values, np.ones(window) / window, mode="valid")


def plot_seed_curves(full_results, key="totals", rolling_window=50, title=None):
    plt.figure(figsize=(12, 6))
    for seed, result in sorted(full_results.items()):
        y = np.asarray(result[key], dtype=np.float64)
        y_smooth = moving_average(y, rolling_window)
        x = np.arange(len(y_smooth))
        if len(y) >= rolling_window:
            x = x + (rolling_window - 1)
        plt.plot(x, y_smooth, label=f"seed {seed}")
    plt.xlabel("Episode")
    plt.ylabel(key)
    plt.title(title or f"{key} by seed")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_mean_std_band(full_results, key="totals", title=None):
    seeds = sorted(full_results.keys())
    min_len = min(len(full_results[s][key]) for s in seeds)
    arr = np.array([
        np.asarray(full_results[s][key][:min_len], dtype=np.float64)
        for s in seeds
    ])
    mean = arr.mean(axis=0)
    std = arr.std(axis=0)
    x = np.arange(min_len)

    plt.figure(figsize=(12, 6))
    plt.plot(x, mean, label="mean")
    plt.fill_between(x, mean - std, mean + std, alpha=0.25, label="±1 std")
    plt.xlabel("Episode")
    plt.ylabel(key)
    plt.title(title or f"{key} mean ± std")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_metric_bars(df, metric="best_100avg", title=None, top_n=10):
    if len(df) == 0:
        print("Empty dataframe.")
        return

    plot_df = df.sort_values(metric, ascending=False).head(top_n).copy()
    labels = []
    for _, row in plot_df.iterrows():
        label = f"id {int(row['config_id'])} / seed {int(row['seed'])}"
        labels.append(label)

    plt.figure(figsize=(12, 6))
    plt.bar(labels, plot_df[metric].values)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel(metric)
    plt.title(title or metric)
    plt.tight_layout()
    plt.show()


def show_top_configs(df, group_cols, metric="best_100avg", top_n=10):
    if len(df) == 0:
        print("Empty dataframe.")
        return pd.DataFrame()

    grouped = (
        df.groupby(group_cols, dropna=False)
        .agg(
            mean_best_100avg=("best_100avg", "mean"),
            std_best_100avg=("best_100avg", "std"),
            mean_last_100=("mean_last_100", "mean"),
            mean_all=("mean_all", "mean"),
            n_runs=("seed", "count"),
        )
        .reset_index()
        .sort_values("mean_best_100avg", ascending=False)
    )
    return grouped.head(top_n)


def plot_config_mean_bands(config_runs, title_prefix="", key="totals"):
    for label, full_results in config_runs.items():
        seeds = sorted(full_results.keys())
        min_len = min(len(full_results[s][key]) for s in seeds)
        arr = np.array([
            np.asarray(full_results[s][key][:min_len], dtype=np.float64)
            for s in seeds
        ])
        mean = arr.mean(axis=0)
        std = arr.std(axis=0)
        x = np.arange(min_len)

        plt.figure(figsize=(12, 6))
        plt.plot(x, mean, label=f"{label} mean")
        plt.fill_between(x, mean - std, mean + std, alpha=0.25)
        plt.xlabel("Episode")
        plt.ylabel(key)
        plt.title(f"{title_prefix}{label}")
        plt.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
reinforce_seeds = [0, 1, 2]

reinforce_grid = list(itertools.product(
    [64],          # size
    [2],            # depth
    [1e-3],      # lr
    [0.95, 0.99],      # discount_factor
))

reinforce_n_episodes = 1500
reinforce_resume = True

In [ ]:

reinforce_rows_all = []
reinforce_full_by_config = {}
reinforce_cfg_table = []

for config_id, (size, depth, lr, discount_factor) in enumerate(reinforce_grid):
    print(f"\n===== REINFORCE config {config_id} =====")
    print(f"size={size}, depth={depth}, lr={lr}, df={discount_factor}")

    rows, full = run_reinforce_seed_sweep(
        seeds=reinforce_seeds,
        size=size,
        depth=depth,
        lr=lr,
        n_episodes=reinforce_n_episodes,
        discount_factor=discount_factor,
        resume=reinforce_resume,
        env_name=ENV_NAME,
        save_root=SAVE_ROOT,
    )

    for row in rows:
        row = row.copy()
        row["config_id"] = config_id
        reinforce_rows_all.append(row)

    reinforce_full_by_config[config_id] = full
    reinforce_cfg_table.append({
        "config_id": config_id,
        "size": size,
        "depth": depth,
        "lr": lr,
        "discount_factor": discount_factor,
    })

reinforce_df = to_df(reinforce_rows_all)
reinforce_cfg_df = pd.DataFrame(reinforce_cfg_table)

display(reinforce_df.head())
display(reinforce_cfg_df)

In [ ]:
# REINFORCE summary table

reinforce_top_configs = show_top_configs(
    reinforce_df,
    group_cols=["config_id", "size", "depth", "lr", "discount_factor"],
    metric="best_100avg",
    top_n=10,
)

display(reinforce_top_configs)

In [ ]:
# REINFORCE plots for top configs

plot_metric_bars(
    reinforce_df,
    metric="best_100avg",
    title="REINFORCE runs ranked by best_100avg",
    top_n=12,
)

top_reinforce_config_ids = reinforce_top_configs["config_id"].head(3).tolist()
top_reinforce_runs = {
    f"cfg {cfg_id}": reinforce_full_by_config[cfg_id]
    for cfg_id in top_reinforce_config_ids
}

plot_config_mean_bands(
    top_reinforce_runs,
    title_prefix="REINFORCE ",
    key="totals",
)

In [ ]:
# inspect one REINFORCE config in detail

cfg_to_plot = int(reinforce_top_configs.iloc[0]["config_id"])

display(reinforce_cfg_df[reinforce_cfg_df["config_id"] == cfg_to_plot])
display(reinforce_df[reinforce_df["config_id"] == cfg_to_plot].sort_values("seed"))

plot_seed_curves(
    reinforce_full_by_config[cfg_to_plot],
    key="totals",
    rolling_window=50,
    title=f"REINFORCE cfg {cfg_to_plot} - reward curves by seed",
)

plot_mean_std_band(
    reinforce_full_by_config[cfg_to_plot],
    key="totals",
    title=f"REINFORCE cfg {cfg_to_plot} - mean ± std reward",
)

In [ ]:
# Actor-Critic sweep config
# AC is slower, so this will likely take the longest

ac_seeds = [0, 1, 2]

ac_grid = list(itertools.product(
    [64],                    # size
    [2],                     # depth
    [1e-4, 3e-4],            # lr
    [0.99],                  # discount_factor
    [0.3, 0.5],              # critic_weight
    [(0.001, 0.0001, 800),   # (ew_start, ew_end, anneal_episodes)
     (0.002, 0.0001, 800)],
))

ac_n_episodes = 3500
ac_resume = True

In [ ]:
# run AC sweep

ac_rows_all = []
ac_full_by_config = {}
ac_cfg_table = []

for config_id, (size, depth, lr, discount_factor, critic_weight, ew_tuple) in enumerate(ac_grid):
    ew_start, ew_end, ew_anneal = ew_tuple

    print(f"\n===== AC config {config_id} =====")
    print(
        f"size={size}, depth={depth}, lr={lr}, df={discount_factor}, "
        f"cw={critic_weight}, ew=({ew_start}, {ew_end}, {ew_anneal})"
    )

    rows, full = run_ac_seed_sweep(
        seeds=ac_seeds,
        size=size,
        depth=depth,
        lr=lr,
        n_episodes=ac_n_episodes,
        discount_factor=discount_factor,
        critic_weight=critic_weight,
        entropy_weight_start=ew_start,
        entropy_weight_end=ew_end,
        entropy_anneal_episodes=ew_anneal,
        resume=ac_resume,
        env_name=ENV_NAME,
        save_root=SAVE_ROOT,
    )

    for row in rows:
        row = row.copy()
        row["config_id"] = config_id
        ac_rows_all.append(row)

    ac_full_by_config[config_id] = full
    ac_cfg_table.append({
        "config_id": config_id,
        "size": size,
        "depth": depth,
        "lr": lr,
        "discount_factor": discount_factor,
        "critic_weight": critic_weight,
        "entropy_weight_start": ew_start,
        "entropy_weight_end": ew_end,
        "entropy_anneal_episodes": ew_anneal,
    })

ac_df = to_df(ac_rows_all)
ac_cfg_df = pd.DataFrame(ac_cfg_table)

display(ac_df.head())
display(ac_cfg_df)

In [ ]:
# AC summary and plots

ac_top_configs = show_top_configs(
    ac_df,
    group_cols=[
        "config_id", "size", "depth", "lr", "discount_factor",
        "critic_weight", "entropy_weight_start", "entropy_weight_end",
        "entropy_anneal_episodes"
    ],
    metric="best_100avg",
    top_n=10,
)

display(ac_top_configs)

plot_metric_bars(
    ac_df,
    metric="best_100avg",
    title="Actor-Critic runs ranked by best_100avg",
    top_n=12,
)

top_ac_config_ids = ac_top_configs["config_id"].head(3).tolist()
top_ac_runs = {
    f"cfg {cfg_id}": ac_full_by_config[cfg_id]
    for cfg_id in top_ac_config_ids
}

plot_config_mean_bands(
    top_ac_runs,
    title_prefix="AC ",
    key="totals",
)

In [ ]:
# inspect one AC config in detail

cfg_to_plot = int(ac_top_configs.iloc[0]["config_id"])

display(ac_cfg_df[ac_cfg_df["config_id"] == cfg_to_plot])
display(ac_df[ac_df["config_id"] == cfg_to_plot].sort_values("seed"))

plot_seed_curves(
    ac_full_by_config[cfg_to_plot],
    key="totals",
    rolling_window=50,
    title=f"AC cfg {cfg_to_plot} - reward curves by seed",
)

plot_mean_std_band(
    ac_full_by_config[cfg_to_plot],
    key="totals",
    title=f"AC cfg {cfg_to_plot} - mean ± std reward",
)

In [ ]:
# compare best REINFORCE config vs best AC config

best_reinforce_cfg = int(reinforce_top_configs.iloc[0]["config_id"])
best_ac_cfg = int(ac_top_configs.iloc[0]["config_id"])

compare_runs = {
    f"REINFORCE cfg {best_reinforce_cfg}": reinforce_full_by_config[best_reinforce_cfg],
    f"AC cfg {best_ac_cfg}": ac_full_by_config[best_ac_cfg],
}

plot_config_mean_bands(
    compare_runs,
    title_prefix="Best config comparison - ",
    key="totals",
)

In [ ]:
# choose the best AC config + best seed, then load its best checkpoint

# best AC config by mean performance across seeds
best_ac_config_id = int(ac_top_configs.iloc[0]["config_id"])

# within that config, pick the best seed by best_100avg
best_ac_run = (
    ac_df[ac_df["config_id"] == best_ac_config_id]
    .sort_values("best_100avg", ascending=False)
    .iloc[0]
)

best_ac_seed = int(best_ac_run["seed"])
best_ac_cfg = ac_cfg_df.loc[ac_cfg_df["config_id"] == best_ac_config_id].iloc[0]

print("Best AC config:")
display(best_ac_cfg.to_frame().T)

print("Best AC seed/run:")
display(best_ac_run.to_frame().T)

In [ ]:
# reconstruct the AC checkpoint path and load the best AC model for guidance

best_ac_dir = get_ac_path(
    discount_factor=float(best_ac_cfg["discount_factor"]),
    critic_weight=float(best_ac_cfg["critic_weight"]),
    size=int(best_ac_cfg["size"]),
    depth=int(best_ac_cfg["depth"]),
    lr=float(best_ac_cfg["lr"]),
    entropy_weight_start=float(best_ac_cfg["entropy_weight_start"]),
    entropy_weight_end=float(best_ac_cfg["entropy_weight_end"]),
    entropy_anneal_episodes=int(best_ac_cfg["entropy_anneal_episodes"]),
    seed=best_ac_seed,
)

best_ac_ckpt = best_ac_dir / "best_actor_critic.pt"
print("Loading AC checkpoint from:", best_ac_ckpt)

best_ac_model, best_ac_model_type = load_model_for_eval(best_ac_ckpt)
print("Loaded model type:", best_ac_model_type)

In [ ]:
# DQN sweep config
# This sweep uses the best AC model as guidance.

dqn_seeds = [0, 1, 2]

dqn_grid = list(itertools.product(
    [64],                  # size
    [2],                   # depth
    [1e-3, 3e-4],          # lr
    [0.99],                # discount_factor
    [50000],               # buffer_size
    [64],                  # batch_size
    [250],                 # target_update_freq
    [1.0],                 # epsilon_start
    [0.05],                # epsilon_end
    [300],                 # epsilon_decay_episodes
    [1000],                # warmup_steps
    [1],                   # train_freq
    [(0.0, 0.0, 1),        # no guidance baseline
     (0.5, 0.0, 150),      # moderate annealed guidance
     (0.8, 0.0, 200)],     # stronger early guidance
))

dqn_n_episodes = 1200
dqn_resume = True

In [ ]:
# run DQN sweep using the best AC model for guidance

dqn_rows_all = []
dqn_full_by_config = {}
dqn_cfg_table = []

for config_id, (
    size, depth, lr, discount_factor, buffer_size, batch_size,
    target_update_freq, epsilon_start, epsilon_end,
    epsilon_decay_episodes, warmup_steps, train_freq, acg_tuple
) in enumerate(dqn_grid):

    ac_guidance_start, ac_guidance_end, ac_guidance_anneal_episodes = acg_tuple

    print(f"\n===== DQN config {config_id} =====")
    print(
        f"size={size}, depth={depth}, lr={lr}, df={discount_factor}, "
        f"buf={buffer_size}, batch={batch_size}, tuf={target_update_freq}, "
        f"eps=({epsilon_start}, {epsilon_end}, {epsilon_decay_episodes}), "
        f"acg=({ac_guidance_start}, {ac_guidance_end}, {ac_guidance_anneal_episodes})"
    )

    rows, full = run_dqn_seed_sweep(
        seeds=dqn_seeds,
        size=size,
        depth=depth,
        lr=lr,
        n_episodes=dqn_n_episodes,
        discount_factor=discount_factor,
        buffer_size=buffer_size,
        batch_size=batch_size,
        target_update_freq=target_update_freq,
        epsilon_start=epsilon_start,
        epsilon_end=epsilon_end,
        epsilon_decay_episodes=epsilon_decay_episodes,
        warmup_steps=warmup_steps,
        train_freq=train_freq,
        resume=dqn_resume,
        ac_model=best_ac_model,
        ac_guidance_start=ac_guidance_start,
        ac_guidance_end=ac_guidance_end,
        ac_guidance_anneal_episodes=ac_guidance_anneal_episodes,
        env_name=ENV_NAME,
        save_root=SAVE_ROOT,
    )

    for row in rows:
        row = row.copy()
        row["config_id"] = config_id
        dqn_rows_all.append(row)

    dqn_full_by_config[config_id] = full
    dqn_cfg_table.append({
        "config_id": config_id,
        "size": size,
        "depth": depth,
        "lr": lr,
        "discount_factor": discount_factor,
        "buffer_size": buffer_size,
        "batch_size": batch_size,
        "target_update_freq": target_update_freq,
        "epsilon_start": epsilon_start,
        "epsilon_end": epsilon_end,
        "epsilon_decay_episodes": epsilon_decay_episodes,
        "warmup_steps": warmup_steps,
        "train_freq": train_freq,
        "ac_guidance_start": ac_guidance_start,
        "ac_guidance_end": ac_guidance_end,
        "ac_guidance_anneal_episodes": ac_guidance_anneal_episodes,
    })

dqn_df = to_df(dqn_rows_all)
dqn_cfg_df = pd.DataFrame(dqn_cfg_table)

display(dqn_df.head())
display(dqn_cfg_df)

In [ ]:
# DQN summary table

dqn_top_configs = show_top_configs(
    dqn_df,
    group_cols=[
        "config_id", "size", "depth", "lr", "discount_factor",
        "buffer_size", "batch_size", "target_update_freq",
        "epsilon_start", "epsilon_end", "epsilon_decay_episodes",
        "ac_guidance_start", "ac_guidance_end", "ac_guidance_anneal_episodes"
    ],
    metric="best_100avg",
    top_n=10,
)

display(dqn_top_configs)

In [ ]:
# DQN plots, especially guidance vs no-guidance

plot_metric_bars(
    dqn_df,
    metric="best_100avg",
    title="DQN runs ranked by best_100avg",
    top_n=12,
)

top_dqn_config_ids = dqn_top_configs["config_id"].head(3).tolist()
top_dqn_runs = {
    f"cfg {cfg_id}": dqn_full_by_config[cfg_id]
    for cfg_id in top_dqn_config_ids
}

plot_config_mean_bands(
    top_dqn_runs,
    title_prefix="DQN ",
    key="totals",
)

In [ ]:
# compare no-guidance vs guided DQN directly

dqn_compare = (
    dqn_df.groupby(
        ["config_id", "ac_guidance_start", "ac_guidance_end", "ac_guidance_anneal_episodes"],
        dropna=False
    )
    .agg(
        mean_best_100avg=("best_100avg", "mean"),
        std_best_100avg=("best_100avg", "std"),
        mean_last_100=("mean_last_100", "mean"),
        n_runs=("seed", "count"),
    )
    .reset_index()
    .sort_values("mean_best_100avg", ascending=False)
)

display(dqn_compare)

In [ ]:
# nspect one DQN config in detail

cfg_to_plot = int(dqn_top_configs.iloc[0]["config_id"])

display(dqn_cfg_df[dqn_cfg_df["config_id"] == cfg_to_plot])
display(dqn_df[dqn_df["config_id"] == cfg_to_plot].sort_values("seed"))

plot_seed_curves(
    dqn_full_by_config[cfg_to_plot],
    key="totals",
    rolling_window=50,
    title=f"DQN cfg {cfg_to_plot} - reward curves by seed",
)

plot_mean_std_band(
    dqn_full_by_config[cfg_to_plot],
    key="totals",
    title=f"DQN cfg {cfg_to_plot} - mean ± std reward",
)

plot_seed_curves(
    dqn_full_by_config[cfg_to_plot],
    key="losses",
    rolling_window=20,
    title=f"DQN cfg {cfg_to_plot} - loss curves by seed",
)

plot_mean_std_band(
    dqn_full_by_config[cfg_to_plot],
    key="losses",
    title=f"DQN cfg {cfg_to_plot} - mean ± std loss",
)

In [ ]:
# Best REINFORCE vs best AC vs best DQN

best_reinforce_cfg = int(reinforce_top_configs.iloc[0]["config_id"])
best_ac_cfg = int(ac_top_configs.iloc[0]["config_id"])
best_dqn_cfg = int(dqn_top_configs.iloc[0]["config_id"])

compare_runs = {
    f"REINFORCE cfg {best_reinforce_cfg}": reinforce_full_by_config[best_reinforce_cfg],
    f"AC cfg {best_ac_cfg}": ac_full_by_config[best_ac_cfg],
    f"DQN cfg {best_dqn_cfg}": dqn_full_by_config[best_dqn_cfg],
}

plot_config_mean_bands(
    compare_runs,
    title_prefix="Best config comparison - ",
    key="totals",
)